In [ ]:
import pandas as pd
import numpy as np

df = pd.read_excel("satisfaction.xlsx")

df = df.drop(columns=["Unnamed: 0"], errors="ignore")

missing_before = (df.isnull().sum() / len(df)) * 100
missing_before_df = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": missing_before
}).sort_values(by="missing_percent", ascending=False)

print("\n===== Missing Values (BEFORE cleaning) =====")
print(missing_before_df[missing_before_df["missing_count"] > 0])

if "Arrival Delay in Minutes" in df.columns:
    df["Arrival Delay in Minutes"] = df["Arrival Delay in Minutes"].fillna(
        df["Arrival Delay in Minutes"].median()
    )

df = df.drop_duplicates()

if "Age" in df.columns:
    df = df[df["Age"] > 0]

if "Departure Delay in Minutes" in df.columns:
    df = df[df["Departure Delay in Minutes"] >= 0]

if "Arrival Delay in Minutes" in df.columns:
    df = df[df["Arrival Delay in Minutes"] >= 0]

categorical_cols = ["Gender", "Customer Type", "Type of Travel", "Class", "satisfaction"]
for col in categorical_cols:
    if col in df.columns:
        df[col] = df[col].astype("category")


if "satisfaction_v2" in df.columns:
    df = df.rename(columns={"satisfaction_v2": "satisfaction"})
df.columns = df.columns.str.strip().str.lower().str.replace(" ", "_")

if "satisfaction" in df.columns:
    # Only works if values match exactly; otherwise you'll see NaNs and can inspect unique values
    df["satisfaction"] = df["satisfaction"].map({
        "neutral or dissatisfied": 0,
        "satisfied": 1
    })

missing_after = (df.isnull().sum() / len(df)) * 100
missing_after_df = pd.DataFrame({
    "missing_count": df.isnull().sum(),
    "missing_percent": missing_after
}).sort_values(by="missing_percent", ascending=False)

print("\n===== Missing Values (AFTER cleaning) =====")
print(missing_after_df[missing_after_df["missing_count"] > 0])

if "id" in df.columns:
    df = df.sort_values(by="id", ascending=True)
else:
    print("\nNo 'id' column found after cleaning. Skipping ID sort.")

output_file = "passenger_satisfaction_cleaned.csv"
df.to_csv(output_file, index=False)

print(f"\nCleaned file exported: {output_file}")
print("Final shape:", df.shape)
